In [2]:
import pandas as pd
import random
import csv
from datasets import load_dataset
from tqdm import tqdm

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

from airouter import AiRouter

client = AiRouter(
   api_key="sk-eXC2ZNKrhHs-9T2Ei1LjOA",
)

In [3]:
# # generated corpus
# tinystories = load_dataset('roneneldan/TinyStories', split='train[:1%]')
# len(tinystories)
# df_tinystories = pd.DataFrame(tinystories['text'], columns=['story'])
# select = [random.randint(0, len(tinystories)) for _ in range(1200)]
# tiny_tinystories = df_tinystories.iloc[select]

In [4]:
tiny_tinystories = pd.read_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/TinyStories/tinystories_part_dutch.csv')

In [5]:
def translate_story(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    prompt = f"""Je krijgt een verhaal in het Engels. Vertaal het zorgvuldig naar het Nederlands. 

Regels:
- Behoud betekenis, stijl en toon.
- Gebruik vloeiend en natuurlijk Nederlands.
- Geen uitleg of extra tekst, alleen de vertaling.

Invoer:
{text}

Uitvoer:
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        weighting={
            "quality": 0.0,
            "latency": 0.0,
        },
        models=['gpt-4o-mini']
    )
    return response.choices[0].message.content.strip()

# Nieuwe kolom met vertalingen
tqdm.pandas()
tiny_tinystories["story_nl_gpt4-o-mini"] = tiny_tinystories["story"].progress_apply(translate_story)


  0%|          | 0/1200 [00:00<?, ?it/s]

100%|██████████| 1200/1200 [1:38:11<00:00,  4.91s/it]


In [6]:
# Opslaan naar CSV
tiny_tinystories.to_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/TinyStories/tinystories_part_dutch.csv', index=False)